# 04 · Bioacoustics

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/microsoft/SPARROW-Playground/blob/main/notebooks/04_bioacoustics.ipynb)

Find bird calls in a recording and name the species. `md-audiobirds-v1` scores each
short window for bird presence; `perch-v2` classifies 5-second windows over about
15,000 species.

The recording is "Yellow-Throated Vireo Song" by SwampVids,
[CC-BY-4.0](https://creativecommons.org/licenses/by/4.0/), from
[Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Vireo_flavifrons_-_Yellow-Throated_Vireo_Song.wav).
Sparrow-Engine reads WAV files.

In [ ]:
# Setup: install sparrow-engine if it is missing (for example on Colab).
# For an NVIDIA GPU, install `sparrow-engine-gpu` instead (same Python API).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("sparrow_engine") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sparrow-engine[video]>=0.1.31"])

import sparrow_engine as spe

print("sparrow-engine", spe.__version__, "| device:", spe.active_device())

In [ ]:
# Demo data: downloaded once into ./data and checked against a published checksum.
import hashlib
import os
import shutil
import urllib.request
import zipfile
from pathlib import Path

DATA = Path(os.environ.get("SPP_DATA_DIR", "data")).resolve()
ASSETS = {
    # Pytorch-Wildlife demo assets, CC-BY-4.0, https://zenodo.org/records/15376499
    "demo_data_det.zip": (
        "https://zenodo.org/records/15376499/files/demo_data_det.zip?download=1",
        "md5", "373e15c8e7c8894379a82086a9177763",
    ),
    "demo_data_base.zip": (
        "https://zenodo.org/records/15376499/files/demo_data_base.zip?download=1",
        "md5", "f89b9b76c753d90e4296d62d71cf1732",
    ),
    # "Yellow-Throated Vireo Song" by SwampVids, CC-BY-4.0, Wikimedia Commons
    "vireo.wav": (
        "https://upload.wikimedia.org/wikipedia/commons/4/4f/"
        "Vireo_flavifrons_-_Yellow-Throated_Vireo_Song.wav",
        "sha1", "2b472209c1b18bfc0c225975838e3a34c1e9887a",
    ),
}


def fetch(name):
    """Download an asset (once), verify it, and unzip archives into DATA."""
    url, algo, digest = ASSETS[name]
    target = DATA / name
    DATA.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        print(f"downloading {name} ...")
        part = target.with_name(target.name + ".part")
        req = urllib.request.Request(url, headers={"User-Agent": "sparrow-playground"})
        with urllib.request.urlopen(req) as r, open(part, "wb") as f:
            shutil.copyfileobj(r, f)
        part.rename(target)
    h = hashlib.new(algo)
    with open(target, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    if h.hexdigest() != digest:
        target.unlink()
        raise RuntimeError(f"{name}: checksum mismatch, deleted; run the cell again")
    marker = DATA / f".{name}.extracted"
    if name.endswith(".zip") and not marker.exists():
        with zipfile.ZipFile(target) as z:
            z.extractall(DATA)
        marker.touch()
    return target

In [ ]:
wav = fetch("vireo.wav")
from IPython.display import Audio
Audio(str(wav))

In [ ]:
# Models are installed into SPARROW_ENGINE_MODEL_DIR (default ~/.sparrow-engine/models).
# Already-installed models are skipped. Inference never downloads anything by itself.
for model_id in ["md-audiobirds-v1", "perch-v2"]:
    spe.download_model(model_id)

## Is there a bird? (`md-audiobirds-v1`)

In [ ]:
presence = spe.detect_audio(wav, model="md-audiobirds-v1")[0]
print(presence)

import matplotlib.pyplot as plt

t = [s.start_time_s for s in presence.segments]
c = [s.confidence for s in presence.segments]
plt.figure(figsize=(10, 2.5))
plt.step(t, c, where="post")
plt.xlabel("seconds"); plt.ylabel("bird presence"); plt.ylim(0, 1.05)

In [ ]:
from IPython.display import Image, display

for png in spe.visualize_audio([presence])[0]:
    display(Image(data=png))

## Which species? (`perch-v2`)

In [ ]:
species = spe.detect_audio(wav, model="perch-v2")[0]
for s in species.segments:
    top_classes = sorted(s.classes, key=lambda c: c.probability, reverse=True)[:3]
    top = ", ".join(f"{c.label} {c.probability:.2f}" for c in top_classes)
    print(f"{s.start_time_s:5.1f}-{s.end_time_s:5.1f}s  {top}")

The first three windows name *Vireo flavifrons*, the Yellow-throated Vireo.
The last window is scored on a partial clip.

For orca calls, try `orca-detector-dclde2026-v5` with `spe.detect_audio_events`;
see notebook 01 for the acoustic models in the zoo.